In [3]:
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd
import requests
from pathlib import Path

In [7]:
df = pd.read_csv('../data/processed/lojas_unificadas_limpas.csv')

In [5]:
pasta_raw = Path('../data/raw')
pasta_raw.mkdir(parents=True, exist_ok=True)
arquivo_ibge_zip = pasta_raw / 'BR_Municipios_2022.zip'

# 2. Download direto do FTP/HTTP oficial do IBGE (Sem bloqueios chatos)
if not arquivo_ibge_zip.exists():
    print("Baixando malha oficial de 2022 direto do IBGE (~80MB)...")
    url_ibge = "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2022/Brasil/BR/BR_Municipios_2022.zip"
    
    # O IBGE é aberto, não precisamos de disfarces complexos aqui
    resposta = requests.get(url_ibge, stream=True)
    resposta.raise_for_status()
    
    with open(arquivo_ibge_zip, 'wb') as f:
        for chunk in resposta.iter_content(chunk_size=8192):
            f.write(chunk)
    print("Download concluído com sucesso!")
else:
    print("Arquivo do IBGE já encontrado localmente.")

Baixando malha oficial de 2022 direto do IBGE (~80MB)...
Download concluído com sucesso!


In [8]:
print("Carregando o mapa na memória...")
# A sintaxe zip:// diz ao geopandas para descompactar e ler na memória
gdf_municipios = gpd.read_file(f"zip://{arquivo_ibge_zip}")

# 4. Converter nossas lojas limpas em um GeoDataFrame
# (Garanta que a variável 'df' da primeira etapa está na memória deste notebook)
gdf_lojas = gpd.GeoDataFrame(
    df, 
    geometry=gpd.points_from_xy(df.lon, df.lat),
    crs="EPSG:4326" 
)

Carregando o mapa na memória...


In [9]:
print("Reprojetando coordenadas...")
gdf_lojas_metric = gdf_lojas.to_crs("EPSG:5880")
gdf_municipios_metric = gdf_municipios.to_crs("EPSG:5880")

Reprojetando coordenadas...


In [10]:
print("Calculando buffers de 30km...")
buffers_30km = gdf_lojas_metric.geometry.buffer(30000)
area_coberta_total = buffers_30km.unary_union

Calculando buffers de 30km...


C:\Users\luizf\AppData\Local\Temp\ipykernel_11844\1459511061.py:3: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  area_coberta_total = buffers_30km.unary_union


In [11]:
print("Cruzando dados espaciais para encontrar Zonas Brancas...")
zonas_brancas_metric = gdf_municipios_metric[gdf_municipios_metric.geometry.disjoint(area_coberta_total)]

Cruzando dados espaciais para encontrar Zonas Brancas...


In [12]:
print(f"Total de municípios no Brasil: {len(gdf_municipios_metric)}")
print(f"Total de municípios em ZONAS BRANCAS (0 lojas num raio de 30km): {len(zonas_brancas_metric)}")

Total de municípios no Brasil: 5572
Total de municípios em ZONAS BRANCAS (0 lojas num raio de 30km): 1368


In [13]:
zonas_brancas_export = zonas_brancas_metric.copy()
zonas_brancas_export['area_km2'] = zonas_brancas_export.geometry.area / 10**6
top_10_oportunidades = zonas_brancas_export.sort_values(by='area_km2', ascending=False).head(10)

In [14]:
print("\n--- TOP 10 ZONAS BRANCAS PARA EXPANSÃO (Por tamanho da área isolada) ---")
print(top_10_oportunidades[['NM_MUN', 'SIGLA_UF', 'area_km2']].to_string(index=False))


--- TOP 10 ZONAS BRANCAS PARA EXPANSÃO (Por tamanho da área isolada) ---
                   NM_MUN SIGLA_UF      area_km2
                 Barcelos       AM 124142.728668
 São Gabriel da Cachoeira       AM 112488.349316
                Oriximiná       PA 107788.332298
                   Tapauá       AM  86519.935276
         Atalaia do Norte       AM  80169.263473
                 Almeirim       PA  72962.711812
                    Jutaí       AM  71659.686340
                   Lábrea       AM  69697.167850
                  Corumbá       MS  64502.413593
Santa Isabel do Rio Negro       AM  64083.692625
